# **EDA Notebook**



---
## 0. Setup Environment

In [46]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [47]:
group_name = "Group 52"
student_name = "Andres Ethorimn Lobarb"
student_id = "26029165"

In [48]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [49]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [50]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [51]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [52]:
import pandas as pd
import altair as alt
import numpy as np
from scipy.stats import norm, spearmanr, t as student_t
from IPython.display import display

In [53]:
# Utils

def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def monthly_mean_change(balances, account_col, date_col, balance_col):
    """Mean month-on-month balance change across accounts, recomputed from balances."""
    ordered = balances.sort_values([account_col, date_col])
    change = ordered.groupby(account_col)[balance_col].diff()
    frame = ordered.assign(change=change).dropna(subset=["change"])
    frame["month"] = pd.to_datetime(frame[date_col]).dt.to_period("M")
    return frame, frame.groupby("month")["change"].mean().rename("mean_change").reset_index()


def lagged_indicators(economics, date_col, columns, lag_months):
    """Indicators keyed by the month they are used in: the row dated `lag_months` earlier."""
    out = economics[[date_col] + columns].copy()
    out["month"] = pd.to_datetime(out[date_col]).dt.to_period("M") + lag_months
    return out.drop(columns=date_col)


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def holm_adjust(p_values):
    """Holm step-down adjustment of a list of p-values (family-wise error control)."""
    order = np.argsort(p_values)
    m = len(p_values)
    adjusted = np.empty(m)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def indicator_scatter(table, indicator, x_title):
    """Monthly mean balance change against a lagged indicator, one point per month."""
    data = table[["month_label", indicator, "mean_change"]]  # Period values are not JSON-serialisable
    return alt.Chart(data).mark_circle(size=60, color="#2a78d6").encode(
        x=alt.X(f"{indicator}:Q", title=x_title, scale=alt.Scale(zero=False)),
        y=alt.Y("mean_change:Q", title="Mean monthly balance change across accounts ($)", scale=alt.Scale(zero=False)),
        tooltip=["month_label", indicator, alt.Tooltip("mean_change:Q", format=".1f")],
    ).properties(width=420, height=260)


def qq_frame(values):
    """Sample quantiles against normal quantiles (all points up to 500 values, else 199 levels).

    Robust standardisation (median, IQR / 1.349) keeps a few extreme values from flattening the plot;
    for normal data it equals the usual standardisation."""
    values = pd.Series(values).dropna()
    n = len(values)
    levels = (np.arange(1, n + 1) - 0.5) / n if n <= 500 else np.linspace(0.005, 0.995, 199)
    scale = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
    scale = scale if scale > 0 else values.std()  # IQR is 0 when most values are equal
    z = (values - values.median()) / scale
    return pd.DataFrame({"theoretical": norm.ppf(levels), "sample": np.quantile(z, levels)})


def distribution_charts(values, label, n_bins, tail):
    """Histogram (the `tail` share at each end not drawn) beside a normal QQ plot, with shape statistics."""
    values = pd.Series(values).dropna()
    low, high = values.quantile(tail), values.quantile(1 - tail)
    shown = values[(values >= low) & (values <= high)]
    counts, edges = np.histogram(shown, bins=n_bins)
    hist = alt.Chart(pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})).mark_bar(
        color="#2a78d6").encode(x=alt.X("start:Q", bin="binned", title=label), x2="end:Q",
                                y=alt.Y("count:Q", title="Count")).properties(width=300, height=220)
    diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
    qq = (alt.Chart(qq_frame(values)).mark_point(color="#2a78d6", size=14).encode(
            x=alt.X("theoretical:Q", title="Normal quantile"), y=alt.Y("sample:Q", title="Robust-standardised sample quantile"))
          + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
          ).properties(width=300, height=220)
    print(f"{label}: n = {len(values)}, skewness = {values.skew():.2f}, excess kurtosis = {values.kurt():.2f}, "
          f"histogram omits {len(values) - len(shown)} values outside the {tail:.1%} to {1 - tail:.1%} quantiles")
    return alt.hconcat(hist, qq)


def group_mean(df, group_col, target):
    """Mean of `target` with a 95% t-interval per group (groups here hold about 12 months)."""
    table = df.groupby(group_col, observed=True)[target].agg(["mean", "std", "size"]).reset_index()
    table = table.rename(columns={"size": "n"})
    half = student_t.ppf(0.975, table["n"] - 1) * table["std"] / np.sqrt(table["n"])
    table["low"], table["high"] = table["mean"] - half, table["mean"] + half
    return table


def quartile_groups(values, decimals):
    """Four equal-size groups of months by value (ties broken by month order), labelled by range."""
    group = pd.qcut(values.rank(method="first"), 4, labels=False)
    ranges = values.groupby(group).agg(["min", "max"])
    labels = {g: f"Q{g + 1}: {r['min']:.{decimals}f} to {r['max']:.{decimals}f}" for g, r in ranges.iterrows()}
    return group.map(labels)


def interval_chart(table, x_field, x_title, y_title, reference, reference_label):
    """Group means with 95% intervals against a grey, labelled reference line.

    The reference is the mean of all months: groups on the line show no effect of the indicator."""
    base = alt.Chart(table).encode(x=alt.X(f"{x_field}:O", title=x_title, sort=None, axis=alt.Axis(labelAngle=0)))
    intervals = base.mark_rule(color="#898781").encode(
        y=alt.Y("low:Q", title=y_title, scale=alt.Scale(zero=False)), y2="high:Q")
    points = base.mark_point(filled=True, color="#2a78d6").encode(
        y="mean:Q", tooltip=[x_field, alt.Tooltip("mean:Q", format=".0f"), alt.Tooltip("low:Q", format=".0f"),
                             alt.Tooltip("high:Q", format=".0f"), "n"])
    ref = pd.DataFrame({"y": [reference], "label": [reference_label]})
    line = alt.Chart(ref).mark_rule(color="#898781", strokeDash=[4, 4]).encode(y="y:Q")
    text = alt.Chart(ref).mark_text(align="left", dx=4, dy=-7, color="#898781").encode(
        y="y:Q", x=alt.value(0), text="label:N")
    return (intervals + points + line + text).properties(width=520, height=260)


In [54]:
# Constants
# Each economic row is dated the first day of its month. A figure for a month is published
# after that month, so the row used for a month is the previous month's (one-month lag).
LAG_MONTHS = 1
INDICATORS = ["inflation_rate", "interest_rate", "unemployment_rate", "consumer_sentiment",
              "loan_stress_index", "spending_factor", "housing_price_index", "equity_market_index"]

---
## B. Data Understanding

**Approach.** We study the indicator table first (B.1), then three indicators chosen in advance for what they mean for deposits (B.2 to B.4), and finally a corrected sweep of every indicator. Each finding comes from the outputs above it; how the economy enters the forecast is decided after the EDA notebooks.

In [55]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "economics.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [56]:
df["economic_date"] = pd.to_datetime(df["economic_date"])
print("Rows, columns:", df.shape)
print("From", df["economic_date"].min().date(), "to", df["economic_date"].max().date(),
      "| all dated the first of a month:", bool((df["economic_date"].dt.day == 1).all()))
print("Months missing in the range:",
      len(pd.date_range(df["economic_date"].min(), df["economic_date"].max(), freq="MS").difference(df["economic_date"])))
display(column_profile(df))
display(df.describe().T.round(3))
print("economic_cycle:", df["economic_cycle"].value_counts().to_dict())
print("recession_flag:", df["recession_flag"].value_counts().to_dict())
cycle_runs = (df["economic_cycle"] != df["economic_cycle"].shift()).cumsum()
print("Consecutive runs of the same cycle phase (months):",
      df.groupby(cycle_runs)["economic_cycle"].agg(["first", "size"]).values.tolist()[:12])

Rows, columns: (72, 11)
From 2020-01-01 to 2025-12-01 | all dated the first of a month: True
Months missing in the range: 0


,type,missing,missing_share,distinct
economic_date,datetime64[ns],0,0.0,72
inflation_rate,float64,0,0.0,53
interest_rate,float64,0,0.0,47
unemployment_rate,float64,0,0.0,40
consumer_sentiment,float64,0,0.0,71
economic_cycle,object,0,0.0,4
loan_stress_index,float64,0,0.0,67
spending_factor,float64,0,0.0,47
housing_price_index,float64,0,0.0,72
equity_market_index,float64,0,0.0,72


,count,mean,min,25%,50%,75%,max,std
economic_date,72,2022-12-16 06:40:00,2020-01-01 00:00:00,2021-06-23 12:00:00,2022-12-16 12:00:00,2024-06-08 12:00:00,2025-12-01 00:00:00,NaN
inflation_rate,72.0,2.301806,1.67,2.0775,2.29,2.4725,3.09,0.315239
interest_rate,72.0,2.862361,2.36,2.6775,2.91,3.0125,3.44,0.26327
unemployment_rate,72.0,5.083056,4.59,5.0,5.11,5.19,5.42,0.152257
consumer_sentiment,72.0,88.288472,69.12,86.1925,92.25,94.475,98.99,10.565051
loan_stress_index,72.0,43.850556,40.36,42.41,43.89,44.8825,47.71,1.748479
spending_factor,72.0,0.658736,0.508,0.64475,0.6875,0.718,0.746,0.080597
housing_price_index,72.0,116.139583,97.8,103.7075,114.825,125.04,142.4,13.131213
equity_market_index,72.0,111.105556,89.02,95.405,103.255,124.83,146.21,18.40231
recession_flag,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


economic_cycle: {'Expansion': 18, 'Peak': 18, 'Slowdown': 18, 'Recovery': 18}
recession_flag: {0: 72}
Consecutive runs of the same cycle phase (months): [['Expansion', 18], ['Peak', 18], ['Slowdown', 18], ['Recovery', 18]]


In [57]:
standardised = df[["economic_date"] + INDICATORS].copy()
standardised[INDICATORS] = (standardised[INDICATORS] - standardised[INDICATORS].mean()) / standardised[INDICATORS].std()
long = standardised.melt("economic_date", var_name="indicator", value_name="z")
long["indicator"] = long["indicator"].str.replace("_", " ").str.capitalize()  # readable legend labels
alt.Chart(long).mark_line().encode(
    x=alt.X("economic_date:T", title="Month"),
    y=alt.Y("z:Q", title="Indicator in standard deviations from its mean"),
    color=alt.Color("indicator:N", title="Indicator"),
    tooltip=["economic_date:T", "indicator", alt.Tooltip("z:Q", format=".2f")],
).properties(width=620, height=260)

alt.Chart(...)

In [58]:
dataset_insights = """
We have 72 complete monthly rows from 2020-01 to 2025-12. Over that time the interest rate moved
between 2.36% and 3.44%, unemployment between 4.59% and 5.42% and consumer sentiment between 69.1
and 99.0; sentiment and <spending_factor> dropped together in a step from January 2023 to July
2024 (chart).

Two columns carry no information: <economic_cycle> is four fixed blocks of 18 months, and
<recession_flag> is always 0.

Because a month's figures are published after it ends, we compare each month's balance change
with the previous month's figures. That leaves 47 months, so the intervals below are wide.
"""

In [59]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `interest_rate`

In [60]:
# Complementary load: how balances moved, month by month, across all accounts
balances = pd.read_csv(at.folder_path / "account_balance_history.csv",
                       usecols=["account_id", "balance_date", "balance"])
account_months, monthly = monthly_mean_change(balances, "account_id", "balance_date", "balance")
table = monthly.merge(lagged_indicators(df, "economic_date", INDICATORS, LAG_MONTHS), on="month", how="inner")
table["month_label"] = table["month"].astype(str)
print("Months with a balance change and a lagged indicator:", len(table),
      "| from", table["month_label"].iloc[0], "to", table["month_label"].iloc[-1])

distribution_charts(table["interest_rate"], "Interest rate, previous month (%)", 12, 0)

Months with a balance change and a lagged indicator: 47 | from 2022-02 to 2025-12
Interest rate, previous month (%): n = 47, skewness = 0.42, excess kurtosis = -0.33, histogram omits 0 values outside the 0.0% to 100.0% quantiles


alt.HConcatChart(...)

In [61]:
display(table["mean_change"].describe().round(1).to_frame("mean monthly change across accounts ($)"))

interest_test = spearman_summary(table["interest_rate"], table["mean_change"], "interest_rate")
display(interest_test.to_frame("Spearman, previous month's interest rate vs mean balance change"))

,mean monthly change across accounts ($)
count,47.0
mean,232.9
std,29.7
min,180.7
25%,206.4
50%,235.5
75%,254.3
max,292.8


,"Spearman, previous month's interest rate vs mean balance change"
feature,interest_rate
n,47
rho,0.3323
ci_low,0.0499
ci_high,0.5655
p_value,0.022493
magnitude,medium


In [62]:
indicator_scatter(table, "interest_rate", "Interest rate, previous month (%)")

alt.Chart(...)

In [63]:
# Does growth rise steadily with the interest rate? Months split into four equal groups by rate.
overall_mean = table["mean_change"].mean()
interest_groups = group_mean(table.assign(group=quartile_groups(table["interest_rate"], 2)), "group", "mean_change")
display(interest_groups.round(1))
interval_chart(interest_groups, "group", "Interest rate group, previous month (%)", "Mean monthly balance change across accounts ($)",
               overall_mean, f"Mean of all months: ${overall_mean:.0f}")

,group,mean,std,n,low,high
0,Q1: 2.67 to 2.89,221.8,30.2,12,202.6,241.0
1,Q2: 2.89 to 2.96,218.6,23.2,12,203.9,233.4
2,Q3: 2.98 to 3.07,255.5,28.7,11,236.2,274.8
3,Q4: 3.09 to 3.44,237.5,24.2,12,222.1,252.9


alt.LayerChart(...)

In [64]:
# Does bank-wide growth trend over time? If growth and an indicator both drifted with the calendar,
# that shared drift alone could produce an association.
month_order = pd.Series(np.arange(len(table)), dtype=float)
display(spearman_summary(month_order, table["mean_change"], "month_order").to_frame(
    "Spearman, month order vs mean balance change"))

over_time = table.assign(month_start=table["month"].dt.to_timestamp())[
    ["month_start", "mean_change", "interest_rate", "unemployment_rate", "consumer_sentiment"]]
growth = alt.Chart(over_time).mark_line(color="#2a78d6", point=True).encode(
    x=alt.X("month_start:T", title=None, axis=alt.Axis(format="%Y-%m", labelAlign="center")), y=alt.Y("mean_change:Q", title="Mean change ($)", scale=alt.Scale(zero=False)))
reference = pd.DataFrame({"y": [overall_mean], "label": [f"Mean of all months: ${overall_mean:.0f}"]})
growth_ref = (alt.Chart(reference).mark_rule(color="#898781", strokeDash=[4, 4]).encode(y="y:Q")
              + alt.Chart(reference).mark_text(align="left", dx=4, dy=-7, color="#898781").encode(
                  y="y:Q", x=alt.value(0), text="label:N"))
rate = alt.Chart(over_time).mark_line(color="#898781", point=True).encode(
    x=alt.X("month_start:T", title="Month", axis=alt.Axis(format="%Y-%m", labelAlign="center")), y=alt.Y("interest_rate:Q", title="Interest rate, previous month (%)",
                                                     scale=alt.Scale(zero=False)))
alt.vconcat((growth + growth_ref).properties(width=520, height=180), rate.properties(width=520, height=120))

,"Spearman, month order vs mean balance change"
feature,month_order
n,47
rho,-0.0393
ci_low,-0.3228
ci_high,0.2507
p_value,0.793031
magnitude,negligible


alt.VConcatChart(...)

In [65]:
feature_1_insights = """
Across all accounts, balances grew by $181 to $293 a month over these 47 months (mean $233). The
interest rate is mildly right-skewed (plots above).

Months that follow a higher interest rate tend to show stronger growth (Spearman rho = 0.33, 95%
interval 0.05 to 0.57), but the scatter is wide and the rise is not steady: the two lower rate
groups sit below the mean of all months ($222 and $219), the two upper groups above it ($256 and
$238).

Growth itself shows no trend over the 47 months (rho = -0.04), while the rate climbs through 2025
(time chart), so a shared trend does not explain the association. We read it as a modest step near
3% rather than a steady effect.
"""

In [66]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `unemployment_rate`

In [67]:
distribution_charts(table["unemployment_rate"], "Unemployment rate, previous month (%)", 12, 0)

Unemployment rate, previous month (%): n = 47, skewness = -0.89, excess kurtosis = 0.79, histogram omits 0 values outside the 0.0% to 100.0% quantiles


alt.HConcatChart(...)

In [68]:
unemployment_test = spearman_summary(table["unemployment_rate"], table["mean_change"], "unemployment_rate")
display(unemployment_test.to_frame("Spearman, previous month's unemployment rate vs mean balance change"))

,"Spearman, previous month's unemployment rate vs mean balance change"
feature,unemployment_rate
n,47
rho,-0.3626
ci_low,-0.5885
ci_high,-0.0842
p_value,0.012254
magnitude,medium


In [69]:
indicator_scatter(table, "unemployment_rate", "Unemployment rate, previous month (%)")

alt.Chart(...)

In [70]:
# Does growth fall steadily as unemployment rises? Months split into four equal groups by rate.
unemployment_groups = group_mean(table.assign(group=quartile_groups(table["unemployment_rate"], 2)), "group",
                                 "mean_change")
display(unemployment_groups.round(1))
interval_chart(unemployment_groups, "group", "Unemployment rate group, previous month (%)", "Mean monthly balance change across accounts ($)",
               overall_mean, f"Mean of all months: ${overall_mean:.0f}")

,group,mean,std,n,low,high
0,Q1: 4.59 to 4.97,246.3,31.0,12,226.6,266.0
1,Q2: 4.98 to 5.11,234.3,31.2,12,214.5,254.1
2,Q3: 5.12 to 5.19,229.9,32.3,11,208.2,251.6
3,Q4: 5.19 to 5.42,220.7,20.6,12,207.6,233.8


alt.LayerChart(...)

In [71]:
feature_2_insights = """
Unemployment is left-skewed (plots above).

Higher unemployment goes with slower growth, a medium negative association (Spearman rho = -0.36,
95% interval -0.59 to -0.08). Unlike the interest rate, the decline is steady across the four
groups, from $246 to $221 a month, although neighbouring intervals overlap. As growth has no trend
over time (B.2), this is not a calendar effect.
"""

In [72]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `consumer_sentiment`

In [73]:
distribution_charts(table["consumer_sentiment"], "Consumer sentiment, previous month (index)", 12, 0)

Consumer sentiment, previous month (index): n = 47, skewness = -0.50, excess kurtosis = -1.80, histogram omits 0 values outside the 0.0% to 100.0% quantiles


alt.HConcatChart(...)

In [74]:
sentiment_test = spearman_summary(table["consumer_sentiment"], table["mean_change"], "consumer_sentiment")
display(sentiment_test.to_frame("Spearman, previous month's consumer sentiment vs mean balance change"))

,"Spearman, previous month's consumer sentiment vs mean balance change"
feature,consumer_sentiment
n,47
rho,0.3312
ci_low,0.0486
ci_high,0.5646
p_value,0.022974
magnitude,medium


In [75]:
indicator_scatter(table, "consumer_sentiment", "Consumer sentiment, previous month (index)")

alt.Chart(...)

In [76]:
# Sentiment is bimodal (plots above), so quartiles would cut across the two clusters. The split is
# the midpoint of the widest gap between consecutive values.
ordered = np.sort(table["consumer_sentiment"].to_numpy())
widest = np.argmax(np.diff(ordered))
split = (ordered[widest] + ordered[widest + 1]) / 2
low = table["consumer_sentiment"] < split
cluster = np.where(low, f"Low: {ordered[0]:.1f} to {ordered[widest]:.1f}",
                   f"High: {ordered[widest + 1]:.1f} to {ordered[-1]:.1f}")
print(f"Split at {split:.1f}; no month lies between {ordered[widest]:.1f} and {ordered[widest + 1]:.1f}")
sentiment_groups = group_mean(table.assign(group=cluster), "group", "mean_change").sort_values("mean")
display(sentiment_groups.round(1))
interval_chart(sentiment_groups, "group", "Consumer sentiment cluster, previous month (index)", "Mean monthly balance change across accounts ($)",
               overall_mean, f"Mean of all months: ${overall_mean:.0f}")

Split at 81.6; no month lies between 72.4 and 90.8


,group,mean,std,n,low,high
1,Low: 69.1 to 72.4,225.1,31.0,18,209.7,240.5
0,High: 90.8 to 93.8,237.7,28.3,29,227.0,248.5


alt.LayerChart(...)

In [77]:
# Holm correction within the three indicators chosen in advance (B.2 to B.4)
chosen = pd.DataFrame([interest_test, unemployment_test, sentiment_test]).set_index("feature")
chosen["p_holm"] = holm_adjust(chosen["p_value"].to_numpy())
display(chosen)

# Exploratory sweep across every indicator, corrected for all eight tests (Holm), and how much
# account-level change the month can explain at most: the share of the variance of account-month
# changes that lies between months.
sweep = pd.DataFrame([spearman_summary(table[c], table["mean_change"], c) for c in INDICATORS]).set_index("feature")
sweep["p_holm"] = holm_adjust(sweep["p_value"].to_numpy())
display(sweep)

grand_mean = account_months["change"].mean()
month_means = account_months.groupby("month")["change"].transform("mean")
between = ((month_means - grand_mean) ** 2).sum()
total = ((account_months["change"] - grand_mean) ** 2).sum()
print("Account-level monthly change, standard deviation ($):", round(account_months["change"].std(), 1))
print("Share of account-month change variance between months (eta squared):", round(between / total, 6))

,n,rho,ci_low,ci_high,p_value,magnitude,p_holm
feature,,,,,,,
interest_rate,47,0.3323,0.0499,0.5655,0.022493,medium,0.044986
unemployment_rate,47,-0.3626,-0.5885,-0.0842,0.012254,medium,0.036763
consumer_sentiment,47,0.3312,0.0486,0.5646,0.022974,medium,0.044986


,n,rho,ci_low,ci_high,p_value,magnitude,p_holm
feature,,,,,,,
inflation_rate,47,-0.2301,-0.4852,0.0611,0.119639,small,0.478556
interest_rate,47,0.3323,0.0499,0.5655,0.022493,medium,0.157452
unemployment_rate,47,-0.3626,-0.5885,-0.0842,0.012254,medium,0.098034
consumer_sentiment,47,0.3312,0.0486,0.5646,0.022974,medium,0.157452
loan_stress_index,47,-0.2278,-0.4834,0.0635,0.123532,small,0.478556
spending_factor,47,0.3210,0.0373,0.5568,0.027807,medium,0.157452
housing_price_index,47,-0.0478,-0.3304,0.2427,0.749902,negligible,1.000000
equity_market_index,47,-0.0661,-0.3467,0.2253,0.658689,negligible,1.000000


Account-level monthly change, standard deviation ($): 4935.7
Share of account-month change variance between months (eta squared): 3.5e-05


In [78]:
feature_n_insights = """
Consumer sentiment falls into two groups of months, around 69 to 73 and 90 to 94.

Months that follow stronger sentiment tend to grow faster (Spearman rho = 0.33, 95% interval 0.05
to 0.56), yet the two clusters differ little: $225 a month after low sentiment against $238 after
high, with overlapping intervals.

These three associations, chosen in advance, hold after Holm correction (p 0.037 to 0.045), while
none of the eight indicators survives the full sweep (smallest p 0.098). Even so, the month
explains only a 0.000035 share of how individual accounts change: the economy moves the bank-wide
average, but hardly tells us anything about a single account.
"""

In [79]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

## C. Ethical Consideration of Features


In [80]:
# No computation: the economic indicators carry no customer attribute; see the text below.

In [81]:
ethical_considerations = """
Economic indicators take the same value for every account, so on their own they cannot treat
customers differently. Downturns, however, hit some groups harder than others, so we audit errors
by customer group, and a macro figure is never used to explain or act on one customer's situation.
"""

In [82]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)